In [1]:
import pandas as pd
from dotenv import load_dotenv
import os
from urllib.parse import quote_plus
from sqlalchemy import create_engine

In [2]:
load_dotenv("../.env")

True

In [3]:
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

In [4]:
password = quote_plus(DB_PASSWORD)

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{password}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("Connected Successfully!")

Connected Successfully!


In [5]:
query = """
SELECT *
FROM telco_customer_churn
LIMIT 2;
"""

pd.read_sql(query, engine)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No


In [6]:
query = """
SELECT COUNT(*) AS total_customers
FROM telco_customer_churn;
"""

pd.read_sql(query, engine)

,total_customers
0,7032


In [7]:
# Churn distribution
query = """
SELECT
    "Churn",
    COUNT(*) AS customer_count
FROM telco_customer_churn
GROUP BY "Churn";
"""

pd.read_sql(query, engine)

,Churn,customer_count
0,No,5163
1,Yes,1869


**Observation:** Out of **7,032 customers**, **1,869 customers have churned**, indicating a substantial churn problem that requires further investigation.

In [8]:
# Average Monthly Charges
query = """
SELECT
ROUND(AVG("MonthlyCharges")::numeric, 2)
AS avg_monthly_charge
FROM telco_customer_churn;
"""

pd.read_sql(query, engine)

,avg_monthly_charge
0,64.8


**Observation:** Customers pay an average monthly charge of approximately **64.80 units**, representing the average recurring revenue generated per customer.

In [9]:
# Contract Distribution
query = """
SELECT
    "Contract",
    COUNT(*) AS customers
FROM telco_customer_churn
GROUP BY "Contract"
ORDER BY customers DESC;
"""

pd.read_sql(query, engine)

,Contract,customers
0,Month-to-month,3875
1,Two year,1685
2,One year,1472


**Observation:** The *Month-to-month contract* is the most common subscription type, indicating that a large proportion of customers prefer flexible plans.

In [15]:
# Average Tenure
query = """
SELECT
    ROUND(AVG("tenure")::numeric, 2)
    AS avg_tenure
FROM telco_customer_churn;
"""

pd.read_sql(query, engine)

,avg_tenure
0,32.42


**Observation:** Customers remain with the company for an average tenure of approximately **32.42 months**, suggesting moderate long-term customer retention.

In [10]:
# Churn Percentage
query = """
SELECT
    "Churn",
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER(),
        2
    ) AS percentage
FROM telco_customer_churn
GROUP BY "Churn";
"""

pd.read_sql(query, engine)

,Churn,percentage
0,No,73.42
1,Yes,26.58


**Observation:** Approximately *26.58% of customers have churned*, while *73.42% remain active*, indicating an imbalanced target variable.

In [17]:
# Churn by Contract Type
query = """
SELECT
    "Contract",
    "Churn",
    COUNT(*) AS customer_count
FROM telco_customer_churn
GROUP BY "Contract", "Churn"
ORDER BY "Contract";
"""

pd.read_sql(query, engine)

,Contract,Churn,customer_count
0,Month-to-month,Yes,1655
1,Month-to-month,No,2220
2,One year,Yes,166
3,One year,No,1306
4,Two year,Yes,48
5,Two year,No,1637


**Observation:** Customers with *Month-to-month contracts exhibit considerably higher churn*, whereas customers with *One-year* and *Two-year contracts demonstrate stronger retention rates*.

In [18]:
# Payment Method Analysis
query = """
SELECT
    "PaymentMethod",
    COUNT(*) AS customer_count
FROM telco_customer_churn
GROUP BY "PaymentMethod"
ORDER BY customer_count DESC;
"""

pd.read_sql(query, engine)

,PaymentMethod,customer_count
0,Electronic check,2365
1,Mailed check,1604
2,Bank transfer (automatic),1542
3,Credit card (automatic),1521


**Observation:** *Electronic check* is the most frequently used payment method among customers, making it the dominant payment channel in the dataset.

In [13]:
# Churn by Internet Service
query = """
SELECT
    "InternetService",
    "Churn",
    COUNT(*) AS customer_count
FROM telco_customer_churn
GROUP BY "InternetService", "Churn"
ORDER BY "InternetService", "Churn";
"""

pd.read_sql(query, engine)

,InternetService,Churn,customer_count
0,DSL,No,1957
1,DSL,Yes,459
2,Fiber optic,No,1799
3,Fiber optic,Yes,1297
4,No,No,1407
5,No,Yes,113


**Observation:** Customers using *Fiber optic internet services exhibit higher churn counts* compared with customers using DSL or having no internet service.

In [14]:
# Senior Citizen Analysis
query = """
SELECT
    "SeniorCitizen",
    COUNT(*) AS customer_count
FROM telco_customer_churn
GROUP BY "SeniorCitizen";
"""

pd.read_sql(query, engine)

,SeniorCitizen,customer_count
0,0,5890
1,1,1142


**Observation:** The customer base primarily consists of *non-senior customers*, while senior citizens represent a smaller proportion of the overall population.


## Technical Summary

- **Libraries Used:** `pandas` was used for data handling and SQL result retrieval, `python-dotenv` for securely loading database credentials, `SQLAlchemy` for creating the PostgreSQL connection, and `psycopg2` as the PostgreSQL database driver.
- **Database Connection:** Connected Python to PostgreSQL using `SQLAlchemy`, `psycopg2`, and environment variables stored in a `.env` file for secure database access.
- **Data Ingestion:** Imported the cleaned customer dataset into PostgreSQL using **Pandas (`df.to_sql()`)**.
- **SQL Operations:** Used `SELECT` to retrieve data, `COUNT()` and `AVG()` to calculate key metrics, `ROUND()` to improve readability, and `GROUP BY` and `ORDER BY` to analyse and organise customer behaviour patterns and churn insights.

---

# Conclusion

The PostgreSQL analysis indicates that customer churn is strongly associated with **contract type**, **internet service category**, and **customer spending behaviour**. These insights provide a strong foundation for subsequent **feature engineering**, **predictive modelling**, and **Customer Lifetime Value (LTV) estimation**.
